# Sanity Checks & Patient/Clinic Trace-Through

This notebook makes the simulation's behavior **inspectable by a human** in two ways:

1. **Boundary condition checks** — Set inputs where the answer is analytically obvious, then verify the simulation produces it. If any of these look wrong, something fundamental is broken.

2. **Step-by-step trace** — Follow a single patient and a single clinic through every timestep of the simulation, watching the state evolve. This catches problems that aggregate metrics hide.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import copy

from sdk.ml.model import ControlledMLModel
from sdk.ml.performance import (
    theoretical_ppv, theoretical_ppv_bounds, check_target_feasibility,
    auc_score, confusion_matrix_metrics,
)
from sdk.core.engine import BranchedSimulationEngine, CounterfactualMode
from sdk.core.scenario import TimeConfig
from scenarios.noshow_overbooking.scenario import (
    ClinicConfig, NoShowOverbookingScenario,
)
from scenarios.stroke_prevention.scenario import (
    StrokeConfig, StrokePreventionScenario,
)

---

# Part 1: Boundary Condition Sanity Checks

## 1.1 Prevalence vs PPV — The Bayes' Theorem Reality Check

The single most important chart for anyone evaluating an ML model in healthcare. Low prevalence *mathematically limits* what PPV is achievable, regardless of how good the model is.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: PPV bounds at different prevalences
for prev, color in [(0.01, 'red'), (0.05, 'orange'), (0.13, 'blue'), (0.30, 'green'), (0.50, 'purple')]:
    sens_range, bounds = theoretical_ppv_bounds(prev, specificities=[0.90])
    axes[0].plot(sens_range, bounds["spec_0.90"], label=f'prev={prev:.0%}', color=color, linewidth=2)
axes[0].set_xlabel('Sensitivity')
axes[0].set_ylabel('Maximum Achievable PPV')
axes[0].set_title('PPV Ceiling at 90% Specificity\n(Bayes\' Theorem)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(0, 1)

# Right: Verify known identities
checks = []
for v in [0.70, 0.80, 0.90, 0.95]:
    ppv = theoretical_ppv(0.50, v, v)
    checks.append((f'50% prev, sens=spec={v:.2f}', v, ppv, abs(ppv - v) < 0.001))

for prev in [0.01, 0.05, 0.13]:
    ppv = theoretical_ppv(prev, 0.80, 1.0)
    checks.append((f'prev={prev:.0%}, spec=1.0', 1.0, ppv, abs(ppv - 1.0) < 0.001))

for prev in [0.05, 0.13, 0.50]:
    ppv = theoretical_ppv(prev, 1.0, 0.0)
    checks.append((f'prev={prev:.0%}, spec=0.0', prev, ppv, abs(ppv - prev) < 0.001))

ppv = theoretical_ppv(0.01, 0.80, 0.99)
checks.append(('1% prev, 99% spec', 0.447, ppv, abs(ppv - 0.447) < 0.01))

colors = ['green' if ok else 'red' for _, _, _, ok in checks]
y_pos = range(len(checks))
axes[1].barh(y_pos, [actual for _, _, actual, _ in checks], color=colors, alpha=0.7, label='Actual')
axes[1].scatter([expected for _, expected, _, _ in checks], y_pos, 
                color='black', marker='|', s=200, linewidths=2, zorder=5, label='Expected')
axes[1].set_yticks(y_pos)
axes[1].set_yticklabels([name for name, _, _, _ in checks], fontsize=8)
axes[1].set_xlabel('PPV')
axes[1].set_title('Bayes\' Theorem Identity Checks\n(green=pass, black marker=expected)')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()
print(f"All {sum(1 for _,_,_,ok in checks if ok)}/{len(checks)} identity checks passed.")

## 1.2 No-Show Scenario at Boundary Conditions

Four boundary tests where the answer is known before running:

| Condition | Expected outcome |
|-----------|-----------------|
| Threshold = 1.0 | Zero overbookings (no predicted prob reaches 1.0) |
| Max individual overbooks = 0 | Zero overbookings (no eligible candidates) |
| Threshold = 1.0 + BRANCHED | Factual == counterfactual (no interventions = identical branches) |
| Low threshold vs high | More overbookings at lower threshold |

In [ ]:
def run_noshow(n_days=15, n_patients=500, seed=42, **kwargs):
    tc = TimeConfig(n_timesteps=n_days, timestep_duration=1/365,
                    timestep_unit="day", prediction_schedule=list(range(n_days)))
    cc = ClinicConfig(n_providers=4, slots_per_provider_per_day=10, max_overbook_per_provider=2)
    defaults = dict(n_patients=n_patients, base_noshow_rate=0.13, model_type="predictor",
                    model_auc=0.83, overbooking_threshold=0.30, clinic_config=cc)
    defaults.update(kwargs)
    sc = NoShowOverbookingScenario(time_config=tc, seed=seed, **defaults)
    return BranchedSimulationEngine(sc, CounterfactualMode.BRANCHED).run(n_patients)

# Test 1: Threshold = 1.0 → zero overbookings
r1 = run_noshow(overbooking_threshold=1.0)
ob1 = r1.outcomes[14].metadata["total_overbooked"]

# Test 2: Max individual = 0 → zero overbookings  
r2 = run_noshow(overbooking_threshold=0.01, max_individual_overbooks=0)
ob2 = r2.outcomes[14].metadata["total_overbooked"]

# Test 3: No intervention → factual == counterfactual
r3 = run_noshow(overbooking_threshold=1.0, n_days=10)
diffs = [np.abs(r3.outcomes[t].events - r3.counterfactual_outcomes[t].events).sum() 
         for t in range(1, 10)]

# Test 4: Monotonicity — lower threshold → more overbooking
r4_low = run_noshow(overbooking_threshold=0.10, n_days=20)
r4_high = run_noshow(overbooking_threshold=0.50, n_days=20)
ob_low = r4_low.outcomes[19].metadata["total_overbooked"]
ob_high = r4_high.outcomes[19].metadata["total_overbooked"]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Test 1
ax = axes[0, 0]
ax.bar(['threshold=1.0', 'expected'], [ob1, 0], color=['blue', 'green'], alpha=0.7)
ax.set_title(f'Threshold=1.0 → Overbookings\nActual: {ob1} (expected: 0)')
ax.set_ylabel('Total overbookings')
ax.annotate('PASS' if ob1 == 0 else 'FAIL', xy=(0.5, 0.5), fontsize=20,
           color='green' if ob1 == 0 else 'red', ha='center', transform=ax.transAxes)

# Test 2  
ax = axes[0, 1]
ax.bar(['max_cap=0', 'expected'], [ob2, 0], color=['blue', 'green'], alpha=0.7)
ax.set_title(f'Max Cap=0 → Overbookings\nActual: {ob2} (expected: 0)')
ax.set_ylabel('Total overbookings')
ax.annotate('PASS' if ob2 == 0 else 'FAIL', xy=(0.5, 0.5), fontsize=20,
           color='green' if ob2 == 0 else 'red', ha='center', transform=ax.transAxes)

# Test 3
ax = axes[1, 0]
ax.bar(range(1, 10), diffs, color=['green' if d == 0 else 'red' for d in diffs])
ax.set_xlabel('Timestep')
ax.set_ylabel('|Factual - CF| total')
ax.set_title(f'No Intervention → F == CF\nMax diff: {max(diffs):.0f} (expected: 0)')
all_zero = all(d == 0 for d in diffs)
ax.annotate('PASS' if all_zero else 'FAIL', xy=(0.5, 0.5), fontsize=20,
           color='green' if all_zero else 'red', ha='center', transform=ax.transAxes)

# Test 4
ax = axes[1, 1]
ax.bar(['thresh=0.10', 'thresh=0.50'], [ob_low, ob_high], color=['blue', 'orange'], alpha=0.7)
ax.set_title(f'Lower Threshold → More Overbooking\n{ob_low} vs {ob_high}')
ax.set_ylabel('Total overbookings')
ok = ob_low >= ob_high
ax.annotate('PASS' if ok else 'FAIL', xy=(0.5, 0.5), fontsize=20,
           color='green' if ok else 'red', ha='center', transform=ax.transAxes)

plt.tight_layout()
plt.show()

---

# Part 2: Step-by-Step Trace-Through

The best way to verify a simulation is correct: follow individual patients and slots through each step and check that every state transition makes sense.

## 2.1 No-Show: Follow a Single Patient Across Days

We'll run the simulation manually (step by step) and track one patient's journey: their no-show probability, whether they were scheduled, whether they were overbooked, whether they showed, and how their history evolves.

In [ ]:
# Set up a small scenario we can step through manually
n_days = 20
tc = TimeConfig(n_timesteps=n_days, timestep_duration=1/365,
                timestep_unit="day", prediction_schedule=list(range(n_days)))
cc = ClinicConfig(n_providers=3, slots_per_provider_per_day=8, max_overbook_per_provider=2)
scenario = NoShowOverbookingScenario(
    time_config=tc, seed=42, n_patients=200,
    base_noshow_rate=0.13, overbooking_threshold=0.25,
    max_individual_overbooks=5, model_type="predictor", model_auc=0.83,
    clinic_config=cc,
)

# Create population and manually run engine loop to capture state at each step
state = scenario.create_population(200)

# Pick a patient with moderate no-show risk to follow
target_patient = None
for pid, p in state.patients.items():
    if 0.10 < p.base_noshow_probability < 0.20:
        target_patient = p
        break
        
print(f"=== TRACKING PATIENT {target_patient.patient_id} ===")
print(f"  Base no-show prob: {target_patient.base_noshow_probability:.3f}")
print(f"  Race/ethnicity:    {target_patient.race_ethnicity}")
print(f"  Insurance:         {target_patient.insurance_type}")
print(f"  Age band:          {target_patient.age_band}")
print(f"  Past appointments: {target_patient.n_past_appointments}")
print(f"  Past no-shows:     {target_patient.n_past_noshows}")
print(f"  Historical rate:   {target_patient.historical_noshow_rate:.3f}")
print()

# Trace the patient through each day
patient_trace = []
pid = target_patient.patient_id

# Fork for CF branch (mimicking engine behavior)
from sdk.core.rng import RNGPartitioner
state_f = state  # factual
state_cf = copy.deepcopy(state)  # counterfactual
factual_rng = scenario.rng
cf_rng = RNGPartitioner(scenario.seed if scenario.seed else 42).create_streams()

for t in range(n_days):
    # Step factual
    scenario.rng = factual_rng
    state_f = scenario.step(state_f, t)
    
    # Step counterfactual
    scenario.rng = cf_rng
    state_cf = scenario.step(state_cf, t)
    
    # Predict + intervene on factual only
    scenario.rng = factual_rng
    predictions = scenario.predict(state_f, t)
    state_f, interventions = scenario.intervene(state_f, predictions, t)
    
    # Record patient state
    p_f = state_f.patients[pid]
    p_cf = state_cf.patients[pid]
    
    # Was patient scheduled today?
    scheduled_f = any(s.patient_id == pid for s in state_f.schedule)
    overbooked_into_f = any(s.overbooked_patient_id == pid for s in state_f.schedule)
    
    # Was patient in a resolved slot? Check resolved_slots
    showed = None
    was_noshow = None
    for s in state_f.resolved_slots:
        if s.patient_id == pid:
            showed = s.original_showed
            was_noshow = not s.original_showed
            break
    
    patient_trace.append({
        'day': t,
        'n_appts_f': p_f.n_past_appointments,
        'n_noshows_f': p_f.n_past_noshows,
        'hist_rate_f': p_f.historical_noshow_rate,
        'n_overbooked_f': p_f.n_times_overbooked,
        'scheduled': scheduled_f,
        'overbooked_into': overbooked_into_f,
        'showed': showed,
        'was_noshow': was_noshow,
        'n_appts_cf': p_cf.n_past_appointments,
        'n_noshows_cf': p_cf.n_past_noshows,
        'n_overbooked_cf': p_cf.n_times_overbooked,
    })

scenario.rng = factual_rng  # restore

# Print the trace
print(f"{'Day':>3s} {'Appts':>5s} {'NoSh':>5s} {'Rate':>6s} {'OB#':>4s} {'Sched':>5s} "
      f"{'OB_in':>5s} {'Show':>5s} {'CF_Ap':>5s} {'CF_NS':>5s} {'CF_OB':>5s}")
print("-" * 70)
for row in patient_trace:
    show_str = {True: 'yes', False: 'NO', None: '-'}[row['showed']]
    print(f"{row['day']:>3d} {row['n_appts_f']:>5d} {row['n_noshows_f']:>5d} "
          f"{row['hist_rate_f']:>6.3f} {row['n_overbooked_f']:>4d} "
          f"{'Y' if row['scheduled'] else '.':>5s} "
          f"{'Y' if row['overbooked_into'] else '.':>5s} "
          f"{show_str:>5s} "
          f"{row['n_appts_cf']:>5d} {row['n_noshows_cf']:>5d} "
          f"{row['n_overbooked_cf']:>4d}")

print("\nKey: Appts=cumulative appointments, NoSh=cumulative no-shows, Rate=historical rate")
print("     OB#=times overbooked, Sched=scheduled today, OB_in=overbooked into a slot")
print("     CF_*=counterfactual branch (no overbooking policy)")

### What to check in the patient trace:
- **Appts should only increase** (never decrease — appointments are cumulative)
- **NoSh should only increase** (cumulative no-shows)
- **Rate = NoSh / Appts** (verify the arithmetic)
- **OB# should only increase on factual, stay 0 on CF** (compounding effect)
- **When scheduled + showed → Appts increments, NoSh stays**
- **When scheduled + NO → Appts increments, NoSh increments**

## 2.2 Follow a Clinic Day: Slot-by-Slot Resolution

Watch every slot in a single day get resolved: who was scheduled, what was predicted, was it overbooked, who showed up, was there a collision?

In [ ]:
# Run a fresh scenario and capture a day's resolved slots
tc2 = TimeConfig(n_timesteps=10, timestep_duration=1/365,
                 timestep_unit="day", prediction_schedule=list(range(10)))
cc2 = ClinicConfig(n_providers=3, slots_per_provider_per_day=6, max_overbook_per_provider=2)
sc2 = NoShowOverbookingScenario(
    time_config=tc2, seed=99, n_patients=150,
    base_noshow_rate=0.13, overbooking_threshold=0.20,
    model_type="predictor", model_auc=0.83, clinic_config=cc2,
)
results2 = BranchedSimulationEngine(sc2, CounterfactualMode.NONE).run(150)

# The engine doesn't expose intermediate state, but we can reconstruct
# from the scenario by re-running manually
sc3 = NoShowOverbookingScenario(
    time_config=tc2, seed=99, n_patients=150,
    base_noshow_rate=0.13, overbooking_threshold=0.20,
    model_type="predictor", model_auc=0.83, clinic_config=cc2,
)
state = sc3.create_population(150)

# Step through 5 days to build up some history
for t in range(5):
    state = sc3.step(state, t)
    preds = sc3.predict(state, t)
    state, intv = sc3.intervene(state, preds, t)

# Now step day 5 → this resolves day 4's schedule
state = sc3.step(state, 5)

# The resolved_slots now contain day 4's fully resolved appointments
print(f"=== CLINIC DAY 4: SLOT-BY-SLOT RESOLUTION ===")
print(f"Slots resolved: {len(state.resolved_slots)}")
print()
print(f"{'Slot':>5s} {'Prov':>4s} {'PatID':>5s} {'TrueP':>6s} {'PredP':>6s} "
      f"{'OB?':>3s} {'OB_Pat':>6s} {'Showed':>6s} {'OB_Sh':>5s} {'Collis':>6s} {'Wait':>5s}")
print("-" * 75)

collisions = 0
noshows = 0
utilized = 0
overbooked = 0

for s in state.resolved_slots:
    is_collision = s.is_overbooked and s.original_showed and s.overbooked_showed
    ob_str = 'yes' if s.is_overbooked else '.'
    ob_pat_str = str(s.overbooked_patient_id) if s.overbooked_patient_id is not None else '.'
    show_str = 'yes' if s.original_showed else 'NO'
    ob_show = 'yes' if s.overbooked_showed else ('NO' if s.is_overbooked else '.')
    coll_str = '!!!' if is_collision else '.'
    wait_str = f'{s.wait_time_minutes:.0f}m' if s.wait_time_minutes > 0 else '.'
    
    print(f"{s.slot_id:>5d} {s.provider_id:>4d} {s.patient_id:>5d} "
          f"{s.true_noshow_prob:>6.3f} {s.predicted_noshow_prob:>6.3f} "
          f"{ob_str:>3s} {ob_pat_str:>6s} {show_str:>6s} {ob_show:>5s} "
          f"{coll_str:>6s} {wait_str:>5s}")
    
    if not s.original_showed:
        noshows += 1
    if s.original_showed or (s.is_overbooked and s.overbooked_showed):
        utilized += 1
    if s.is_overbooked:
        overbooked += 1
    if is_collision:
        collisions += 1

n_slots = len(state.resolved_slots)
print()
print(f"Summary: {n_slots} slots, {noshows} no-shows ({noshows/max(n_slots,1):.0%}), "
      f"{overbooked} overbooked, {collisions} collisions, "
      f"utilization {utilized/max(n_slots,1):.0%}")

### What to check in the slot trace:
- **TrueP** (true no-show prob) should be near the patient's base rate (plus small variability)
- **PredP** (predicted) should correlate with TrueP but with noise — not identical
- **OB?=yes** only when PredP exceeds the overbooking threshold (0.20)
- **Showed=NO** patients should have higher TrueP on average
- **Collisions (!!!)** happen when OB?=yes AND both original and overbooked showed
- **Wait time** > 0 only on collisions
- **No-show rate** should be near 13% population average

## 2.3 Overbooking Burden: Factual vs Counterfactual Over Time

The compounding effect visualized: watch how overbooking burden accumulates day by day on the factual branch, while the counterfactual stays at zero.

In [ ]:
# Run a longer simulation to see compounding
r_long = run_noshow(n_days=30, n_patients=500, overbooking_threshold=0.25)

days = list(range(30))
f_burden = [r_long.outcomes[t].metadata["mean_overbooking_burden"] for t in days]
cf_burden = [r_long.counterfactual_outcomes[t].metadata["mean_overbooking_burden"] for t in days]
f_collisions = [r_long.outcomes[t].metadata["total_collisions"] for t in days]
f_overbooked = [r_long.outcomes[t].metadata["total_overbooked"] for t in days]

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# Overbooking burden
ax = axes[0, 0]
ax.plot(days, f_burden, 'b-o', markersize=3, label='Factual (with overbooking)')
ax.plot(days, cf_burden, 'r--s', markersize=3, label='Counterfactual (no overbooking)')
ax.set_xlabel('Day')
ax.set_ylabel('Mean overbooking burden per patient')
ax.set_title('Compounding Overbooking Burden')
ax.legend()
ax.grid(True, alpha=0.3)

# Cumulative collisions
ax = axes[0, 1]
ax.plot(days, f_collisions, 'orange', linewidth=2)
ax.set_xlabel('Day')
ax.set_ylabel('Cumulative collisions')
ax.set_title('Cumulative Collisions Over Time')
ax.grid(True, alpha=0.3)

# Cumulative overbookings
ax = axes[1, 0]
ax.plot(days, f_overbooked, 'green', linewidth=2)
ax.set_xlabel('Day')
ax.set_ylabel('Cumulative overbookings')
ax.set_title('Cumulative Overbookings')
ax.grid(True, alpha=0.3)

# Collision rate = collisions / overbookings
ax = axes[1, 1]
coll_rate = [c / max(o, 1) for c, o in zip(f_collisions, f_overbooked)]
ax.plot(days, coll_rate, 'red', linewidth=2)
ax.set_xlabel('Day')
ax.set_ylabel('Collision rate')
ax.set_title('Running Collision Rate\n(should stabilize as N grows)')
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1)

plt.tight_layout()
plt.show()

print(f"Day 30 factual burden:  {f_burden[-1]:.3f} (should be > 0)")
print(f"Day 30 CF burden:       {cf_burden[-1]:.3f} (should be exactly 0)")
print(f"Total collisions:       {f_collisions[-1]}")
print(f"Total overbookings:     {f_overbooked[-1]}")
print(f"Final collision rate:   {coll_rate[-1]:.3f}")

## 2.4 Stroke: Follow a Patient's Risk Trajectory

Watch a single patient's risk evolve through the AR(1) process, see when they're flagged by the model, and track the intervention's effect on their risk.

In [ ]:
from scenarios.stroke_prevention.scenario import BASE_RISKS, AR1_MODS, TX_EFFECT, CURRENT_RISKS

config = StrokeConfig(n_patients=500, n_weeks=52, prediction_interval=4,
                      intervention_effectiveness=0.50)
sc_stroke = StrokePreventionScenario(config=config, seed=42)
state_s = sc_stroke.create_population(500)

# Pick a medium-risk patient
patient_idx = np.argsort(state_s[BASE_RISKS])[-50]  # 50th highest risk
print(f"=== TRACKING STROKE PATIENT {patient_idx} ===")
print(f"  Base annual risk: {state_s[BASE_RISKS, patient_idx]:.4f}")
print()

# Manually step through, recording state
stroke_trace = []
for t in range(52):
    state_s = sc_stroke.step(state_s, t)
    
    predicted = False
    score = None
    treated = False
    if t in sc_stroke.time_config.prediction_schedule:
        preds = sc_stroke.predict(state_s, t)
        state_s, intv = sc_stroke.intervene(state_s, preds, t)
        predicted = True
        score = preds.scores[patient_idx]
        treated = patient_idx in intv.treated_indices
    
    outcomes = sc_stroke.measure(state_s, t)
    had_event = outcomes.events[patient_idx] > 0
    
    stroke_trace.append({
        'week': t,
        'base_risk': state_s[BASE_RISKS, patient_idx],
        'ar1_mod': state_s[AR1_MODS, patient_idx],
        'tx_effect': state_s[TX_EFFECT, patient_idx],
        'current_risk': state_s[CURRENT_RISKS, patient_idx],
        'predicted': predicted,
        'score': score,
        'treated': treated,
        'event': had_event,
    })

# Plot the patient journey
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

weeks = [r['week'] for r in stroke_trace]

# Risk trajectory
ax = axes[0]
ax.plot(weeks, [r['current_risk'] for r in stroke_trace], 'b-', linewidth=1.5, label='Current risk')
ax.plot(weeks, [r['base_risk'] for r in stroke_trace], 'gray', linewidth=1, alpha=0.5, label='Base risk')
ax.axhline(y=state_s[BASE_RISKS, patient_idx], color='gray', linestyle=':', alpha=0.3)
# Mark treatment events
treat_weeks = [r['week'] for r in stroke_trace if r['treated']]
treat_risks = [r['current_risk'] for r in stroke_trace if r['treated']]
ax.scatter(treat_weeks, treat_risks, color='green', s=60, zorder=5, marker='v', label='Treated')
# Mark stroke events
event_weeks = [r['week'] for r in stroke_trace if r['event']]
event_risks = [r['current_risk'] for r in stroke_trace if r['event']]
ax.scatter(event_weeks, event_risks, color='red', s=100, zorder=5, marker='*', label='Stroke event')
ax.set_ylabel('Annual risk')
ax.set_title(f'Patient {patient_idx}: Risk Trajectory Over 52 Weeks')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# AR(1) modifier and treatment effect
ax = axes[1]
ax.plot(weeks, [r['ar1_mod'] for r in stroke_trace], 'purple', linewidth=1.5, label='AR(1) modifier')
ax.plot(weeks, [r['tx_effect'] for r in stroke_trace], 'green', linewidth=1.5, label='Treatment effect')
ax.axhline(y=1.0, color='gray', linestyle=':', alpha=0.3)
ax.set_ylabel('Modifier value')
ax.set_title('Temporal Modifier & Treatment Effect')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Prediction scores at prediction times
ax = axes[2]
pred_weeks = [r['week'] for r in stroke_trace if r['predicted']]
pred_scores = [r['score'] for r in stroke_trace if r['predicted']]
ax.bar(pred_weeks, pred_scores, width=3, alpha=0.7, color='orange', label='Prediction score')
ax.axhline(y=config.treatment_threshold, color='red', linestyle='--', 
           alpha=0.5, label=f'Treatment threshold ({config.treatment_threshold})')
ax.set_xlabel('Week')
ax.set_ylabel('Prediction score')
ax.set_title('ML Model Predictions (every 4 weeks)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print key moments
print("\nKey events:")
for r in stroke_trace:
    if r['treated']:
        print(f"  Week {r['week']:2d}: TREATED (score={r['score']:.3f}), "
              f"risk dropped from {r['current_risk']/(1-config.intervention_effectiveness):.4f} "
              f"to {r['current_risk']:.4f}")
    if r['event']:
        print(f"  Week {r['week']:2d}: STROKE EVENT at risk={r['current_risk']:.4f}")

### What to check in the stroke trace:
- **Current risk = base_risk x AR(1) modifier x treatment effect** (verify the arithmetic visually)
- **AR(1) modifier** should be autocorrelated (smooth, not jagged) and mean-revert toward 1.0
- **Treatment effect** drops from 1.0 to 0.5 when treated — and STAYS at 0.5 (persistent)
- **Current risk drops** at treatment point and stays lower (visible step-down in top panel)
- **Prediction scores** should roughly correlate with risk (higher risk = higher score, with noise)
- **Treatment only when score exceeds threshold** (bottom panel: bars above red line)
- **Stroke events** (red stars) should be rare and more likely at higher risk levels